Train Ultralytics YOLOv8n on GPU (Google Colab Free T4) for 30 epochs to generate complete loss curves and eliminate the 3-epoch CPU limitation.

### Pipeline Steps:
1. Setup GPU Environment & Install Ultralytics
2. Download / Mount University of Peshawar Whiteboard Dataset (1,964 images)
3. Execute 30-Epoch Controlled Training with Mosaic & Geometric Augmentations
4. Generate CIoU Box Loss, BCE Classification Loss, DFL Loss, and mAP@50-95 Curves
5. Evaluate on Held-Out Test Split (177 images)
6. Download Champion `whiteboard_detector_best.pt` for BoardMind AI

In [1]:
# Step 1: Verify GPU & Install Ultralytics
!nvidia-smi
!pip install -q ultralytics

Sun Nov 15 15:47:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.91                 KMD Version: 610.91        CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   38C    P0              6W /   70W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Step 2: Download the University of Peshawar Whiteboard Dataset via Roboflow
# (Or upload your local datasets/whiteboard zip)
!pip install -q roboflow
from roboflow import Roboflow

# You can also use curl/gdown or Roboflow Universe public download:
# rf = Roboflow(api_key="YOUR_ROBOFLOW_KEY_IF_ANY")
# project = rf.workspace("university-of-peshawar-4qet5").project("whiteboard-dataset")
# dataset = project.version(1).download("yolov8")

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\skand\\anaconda3\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'roboflow'

In [ ]:
# Step 3: Train YOLOv8n Champion Model (30 Epochs on GPU)
from ultralytics import YOLO

# Initialize from COCO pretrained weights
model = YOLO("yolov8n.pt")

results = model.train(
    data="datasets/whiteboard/data.yaml",  # Path to data.yaml
    epochs=30,
    imgsz=640,
    batch=16,
    device=0,  # GPU
    optimizer="auto",
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=2.0,
    # Controlled Augmentations to Break Full-Frame Bias
    degrees=10.0,
    translate=0.15,
    scale=0.6,
    perspective=0.001,
    mosaic=1.0,
    fliplr=0.5,
    project="runs/whiteboard_gpu",
    name="exp_champion_30epochs",
    exist_ok=True
)

In [ ]:
# Step 4: Evaluate on Untouched Test Split
best_weights = "runs/whiteboard_gpu/exp_champion_30epochs/weights/best.pt"
trained_model = YOLO(best_weights)

test_metrics = trained_model.val(
    data="datasets/whiteboard/data.yaml",
    split="test",
    imgsz=640,
    batch=16
)

print("=== FINAL TEST METRICS ===")
print(f"mAP@50:    {test_metrics.box.map50 * 100:.2f}%")
print(f"mAP@50-95: {test_metrics.box.map * 100:.2f}%")
print(f"Precision: {test_metrics.box.mp * 100:.2f}%")
print(f"Recall:    {test_metrics.box.mr * 100:.2f}%")

In [ ]:
# Step 5: Visualize Training Curves (Results & Confusion Matrix)
from IPython.display import Image, display

display(Image("runs/whiteboard_gpu/exp_champion_30epochs/results.png"))
display(Image("runs/whiteboard_gpu/exp_champion_30epochs/confusion_matrix.png"))

In [ ]:
# Step 6: Download Champion Weights to place in models/whiteboard_detector_best.pt
from google.colab import files
files.download(best_weights)